<a href="https://colab.research.google.com/github/GuardAzul/ModeloLSCTrain/blob/main/ModeloLSCTrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.regularizers import l2
from keras.utils import to_categorical
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import json
import numpy as np
import os
import shutil
from google.colab import drive
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [ ]:
# Montar Google Drive
drive.mount('/content/drive')

# Definir las rutas
RESOURSES_PATH = "/content/resourses"
MODEL_FRAMES = 60
MODEL_FOLDER_PATH = os.path.join(RESOURSES_PATH, "model")
KEYPOINTS_PATH = os.path.join(RESOURSES_PATH, "keypoints")
MODEL_PATH = os.path.join(MODEL_FOLDER_PATH, f"actions_{MODEL_FRAMES}.keras")
WORDS_JSON_PATH = os.path.join(RESOURSES_PATH, "words.json")
LENGTH_KEYPOINTS = 1662

# Ruta a la carpeta en Google Drive
drive_folder_path = "/content/drive/My Drive/resources_ia_lsc"

# Crear las carpetas necesarias si no existen
os.makedirs(MODEL_FOLDER_PATH, exist_ok=True)
os.makedirs(KEYPOINTS_PATH, exist_ok=True)

# Cargar el archivo words.json desde Google Drive
words_json_source = os.path.join(drive_folder_path, "words.json")

# Ruta de la carpeta en Google Drive
drive_folder_path = "/content/drive/My Drive/resources_ia_lsc"

shutil.copy(words_json_source, WORDS_JSON_PATH)

# Cargar todos los archivos .h5 desde la carpeta de Drive a KEYPOINTS_PATH
for file_name in os.listdir(drive_folder_path):
    if file_name.endswith(".h5"):
        file_source = os.path.join(drive_folder_path, file_name)
        file_dest = os.path.join(KEYPOINTS_PATH, file_name)
        shutil.copy(file_source, file_dest)

if os.path.exists(WORDS_JSON_PATH):
    print("words.json copiado correctamente")
else:
    print("Error al copiar words.json")

h5_files = [f for f in os.listdir(KEYPOINTS_PATH) if f.endswith('.h5')]
if h5_files:
    print(f"Se copiaron {len(h5_files)} archivos .h5 correctamente")
else:
    print("No se encontraron archivos .h5 en la carpeta")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
words.json copiado correctamente
Se copiaron 4 archivos .h5 correctamente


In [ ]:
def get_word_ids(path):
    with open(path, 'r') as json_file:
        data = json.load(json_file)
        print(data)
        return data.get('word_ids')

In [ ]:
def get_sequences_and_labels(words_id):
    sequences, labels = [], []
    for word_index, word_id in enumerate(words_id):
        hdf_path = os.path.join(KEYPOINTS_PATH, f"{word_id}.h5")
        try:
            # cargar los datos
            data = pd.read_hdf(hdf_path, key='data')
            # verificar si el dataframe está vacío
            if data.empty:
                print(f"Warning: DataFrame for {hdf_path} is empty.")
                continue

            # Verificar la forma de los datos
            if data.shape[1] != LENGTH_KEYPOINTS:
                print(f"Warning: El archivo {hdf_path} tiene {data.shape[1]} keypoints, pero se espereaban {LENGTH_KEYPOINTS}.")
                continue

            # Convert the data to a list of frames
            seq_keypoints = data.values.tolist()
            sequences.append(seq_keypoints)
            labels.append(word_index)

        except Exception as e:
            print(f"Error loading {hdf_path}: {e}")
        except KeyError:
            print(f"Error: Clave 'data' no encontrada en {hdf_path}.")
        except pd.errors.HDF5Error:
            print(f"Error: Formato de archivo HDF5 inválido en {hdf_path}.")
        except Exception as e:
            print(f"Error inesperado al cargar {hdf_path}: {e}")

    return sequences, labels

In [ ]:
# Load and inspect a sa
sample_hdf_path = os.path.join(KEYPOINTS_PATH, 'buenos_dias.h5')

try:
  # Cargar datos
  data = pd.read_hdf(sample_hdf_path, key='data')

  # Verificar la estructura de los datos
  print("Columns:", data.columns)
  print("Número de columnas: ", len(data.columns))
  print("Número de filas: ", len(data))

  # Mostrar las primeras filas
  print(data.head())

  # Verificar valores faltantes
  print("Valores faltantes por columna: ")
  print(data.isnull().sum())

  # verificar rangos de valores
  print("Rango de valores por columna: ")
  print(data.describe())

except KeyError:
    print(f"Error: La clave 'data' no está presente en {sample_hdf_path}.")
except FileNotFoundError:
    print(f"Error: Archivo no encontrado en {sample_hdf_path}.")
except Exception as e:
  print(f"Error loading {sample_hdf_path}: {e}")


Columns: Index([   0,    1,    2,    3,    4,    5,    6,    7,    8,    9,
       ...
       1652, 1653, 1654, 1655, 1656, 1657, 1658, 1659, 1660, 1661],
      dtype='int64', length=1662)
Número de columnas:  1662
Número de filas:  4016
       0         1         2         3         4         5         6     \
0  0.507430  0.422963 -0.801830  0.999973  0.522057  0.363936 -0.750732   
1  0.495359  0.400714 -0.806467  0.999981  0.520173  0.341918 -0.750671   
2  0.498601  0.398787 -0.921763  0.999972  0.520143  0.335715 -0.870990   
3  0.498454  0.404003 -0.913456  0.999970  0.520179  0.349036 -0.867187   
4  0.501270  0.401829 -0.754356  0.999952  0.523546  0.337539 -0.708520   

       7         8         9     ...      1652      1653      1654      1655  \
0  0.999959  0.535010  0.363108  ... -0.054147  0.499349  0.888100 -0.062052   
1  0.999972  0.533972  0.342502  ... -0.050275  0.518527  0.779582 -0.064852   
2  0.999955  0.533317  0.334560  ... -0.042391  0.529925  0.831604 -0.0

Despúes de analizar los datos obtenidos, se normalizaran los datos

In [ ]:
from sklearn.preprocessing import StandardScaler

# Aplicar estandarización a todas las columnas
scaler = StandardScaler()
data_normalized = data.copy()
data_normalized[data.columns] = scaler.fit_transform(data[data.columns])

data_normalized = pd.DataFrame(data_normalized, columns=data.columns)

# Verificar los nuevos rangos
print("Rango de valores normalizados por columna: ")
print(data_normalized.describe())


Rango de valores normalizados por columna: 
               0             1             2             3             4     \
count  4.016000e+03  4.016000e+03  4.016000e+03  4.016000e+03  4.016000e+03   
mean   6.440178e-16 -5.661695e-16 -1.415424e-16 -3.324565e-13 -1.132339e-15   
std    1.000125e+00  1.000125e+00  1.000125e+00  1.000125e+00  1.000125e+00   
min   -3.274232e+00 -3.013110e+00 -5.280376e+00 -3.973199e+01 -3.031385e+00   
25%   -5.928410e-01 -3.989742e-01 -4.175661e-01  5.182509e-02 -6.140965e-01   
50%   -1.779491e-02  1.356133e-01  2.248274e-01  1.957987e-01 -1.974915e-02   
75%    7.125431e-01  6.550148e-01  6.672190e-01  2.570641e-01  7.063029e-01   
max    2.981783e+00  2.744753e+00  2.320915e+00  2.945891e-01  2.790060e+00   

               5             6             7             8             9     \
count  4.016000e+03  4.016000e+03  4.016000e+03  4.016000e+03  4.016000e+03   
mean   5.661695e-16  2.830848e-17  1.841696e-13 -3.043161e-16  2.264678e-16   
std    

In [ ]:
from re import S
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout


''' # Dividir los datos en conjuntos de entrenamiento
X_train, X_test, y_train, y_test = train_test_split(data_normalized, data_normalized, test_size=0.2, random_state=42) '''

''' def get_model(input_length, num_classes):
    model = Sequential()
    model.add(LSTM(32, input_shape=(input_length, LENGTH_KEYPOINTS), return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(num_classes, activation='softmax'))
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model '''

def get_model(input_length, num_classes, labels):
    model = Sequential()
    model.add(LSTM(128, input_shape=(input_length, LENGTH_KEYPOINTS), return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(num_classes, activation='relu'))
    model.add(Dense(len(set(labels)), activation='softmax'))
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
def training_model(model_path, epochs=100):
    # Ensure the word IDs are retrieved properly
    word_ids = get_word_ids(WORDS_JSON_PATH) # ['word1', 'word2', 'word3']

    sequences, labels = get_sequences_and_labels(word_ids)

    # Verificar los valores en labels
    print("Valores únicos en labels:", np.unique(labels))

    # Verificar las secuencias generadas
    print("Número de secuencias:", len(sequences))
    print("Longitudes de las secuencias:", [len(seq) for seq in sequences])

    # Pad sequences
    sequences = pad_sequences(sequences, maxlen=int(MODEL_FRAMES), padding='pre', truncating='post', dtype='float16')

    X = np.array(sequences)
    y = to_categorical(labels, num_classes=len(word_ids))

    # Check dataset sizes
    print(f"Dataset size: X={X.shape}, y={y.shape}")

    # Handle small datasets
    if len(X) <= 1:
        raise ValueError("Insufficient samples to split into training and validation sets.")

    # Train-test split
    try:
        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)
    except ValueError as e:
        print(f"Error in train_test_split: {e}")
        return

    # Get the model
    model = get_model(int(MODEL_FRAMES), len(word_ids), labels)

    # Early stopping callback
   # early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

   # Model training
    try:
        # Entrenamiento sin EarlyStopping
        model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=epochs, batch_size=8)
    except Exception as e:
        print(f"Error during model training: {e}")
        return


    # Verifica las dimensiones después de la división
    print(f"Training set size: {X_train.shape[0]} samples")
    print(f"Validation set size: {X_val.shape[0]} samples")

    # Model summary
    model.summary()

    # Save model
    try:
        model.save(model_path)
        print(f"Model saved to {model_path}")
    except Exception as e:
        print(f"Error saving model: {e}")

    # Evaluate the model on validation set to get predictions
    y_pred = model.predict(X_val)

    # Asegúrate de que las predicciones sean índices de clases
    # Si las etiquetas están en one-hot encoded, usamos np.argmax() para convertirlas a índices de clase
    y_pred_classes = np.argmax(y_pred, axis=1)  # Convert predictions to class indices
    y_true_classes = np.argmax(y_val, axis=1)   # Get true class indices (if one-hot encoded)

    # Generate the confusion matrix
    cm = confusion_matrix(y_true_classes, y_pred_classes)

    # Imprimir la matriz de confusión
    print("Confusion Matrix:")
    print(cm)

    # Si deseas visualizar la matriz de confusión (opcional)
    import seaborn as sns
    import matplotlib.pyplot as plt

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=word_ids, yticklabels=word_ids)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

# Function to load and verify the model
def load_and_verify_model(model_path):
    try:
        model = load_model(model_path)
        model.summary()
        print("Model loaded and verified successfully.")
    except Exception as e:
        print(f"Error loading model: {e}")

In [ ]:
training_model(MODEL_PATH)

{'word_ids': ['buenos_dias', 'como_estas', 'gracias']}
Error loading /content/resourses/keypoints/como_estas.h5: HDF5 error back trace

  File "H5F.c", line 836, in H5Fopen
    unable to synchronously open file
  File "H5F.c", line 796, in H5F__open_api_common
    unable to open file
  File "H5VLcallback.c", line 3863, in H5VL_file_open
    open failed
  File "H5VLcallback.c", line 3675, in H5VL__file_open
    open failed
  File "H5VLnative_file.c", line 128, in H5VL__native_file_open
    unable to open file
  File "H5Fint.c", line 2018, in H5F_open
    unable to read superblock
  File "H5Fsuper.c", line 392, in H5F__super_read
    file signature not found

End of HDF5 error back trace

Unable to open/create file '/content/resourses/keypoints/como_estas.h5'
Valores únicos en labels: [0 2]
Número de secuencias: 2
Longitudes de las secuencias: [4016, 11493]
Dataset size: X=(2, 60, 1662), y=(2, 3)
Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Error during model training: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 3), output.shape=(None, 2)


In [ ]:
load_and_verify_model(MODEL_PATH)

Error loading model: name 'load_model' is not defined
